# 🎓 KORA-lite — Notebook 5 (bonus) : entraîner NOTRE oreille fon sur J'aime Ma Langue

Dans le notebook 3, on a **utilisé** le modèle `Professor/mms-300m-fongbe` tel quel. Ici, on **l'entraîne nous-mêmes** sur les enregistrements **J'aime Ma Langue**, faits au téléphone par des Béninois.

**Deux options, à choisir dans la cellule de réglages :**

| | Option A — « Adapter » | Option B — « Notre propre oreille » |
|---|---|---|
| Point de départ | `Professor/mms-300m-fongbe` (connaît déjà le fon) | `facebook/wav2vec2-xls-r-300m` (n'a jamais entendu de fon) |
| Ce qu'on fait | On l'**adapte** à la voix au téléphone | On lui **apprend le fon** avec J'aime Ma Langue |
| Résultat attendu | Meilleur que le modèle de départ sur des voix de téléphone | Moins bon au début : il lui faut beaucoup plus de données et de temps |
| Durée (T4) | ~30 min | ~1 h 15 |
| Licence du résultat | ❌ non commerciale (hérite de CC-BY-NC) | ✅ **commercialisable** (Apache 2.0 + J'aime Ma Langue CC-BY-SA, en citant) |
| Pour le pitch | « On a amélioré l'oreille fon sur des voix de téléphone » | « On a entraîné **notre propre** reconnaissance vocale fon » |

💡 **Conseil** : lancez l'**option A** pour la démo. Si vous avez du temps de GPU en plus pendant le challenge, lancez l'**option B** : même avec un score modeste, c'est la preuve de la feuille de route commerciale.

⚠️ **Ce notebook est le plus expérimental du kit.** Il suit la recette standard de Hugging Face pour entraîner un modèle vocal (CTC), mais il n'a pas pu être testé avant le challenge. Si une cellule plante, copiez le message d'erreur et demandez de l'aide.

⚙️ GPU T4 obligatoire. Gardez l'onglet Colab ouvert pendant l'entraînement.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip -q install -U transformers accelerate jiwer

In [ ]:
# ▶ Connexion à Google Drive et chemins du projet
# En Colab : le dossier KORA doit être dans "Mon Drive" (MyDrive/KORA).
# En local : le notebook utilise le dossier parent.
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/KORA"
else:
    BASE = os.path.abspath("..")
sys.path.append(BASE)
for d in ["data", "modeles", "resultats"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
print("Dossier du projet :", BASE)
print("Contenu :", sorted(os.listdir(BASE)))

## 1. Les réglages

In [ ]:
OPTION = "A"                 # "A" = adapter mms-300m-fongbe ; "B" = notre propre oreille depuis XLS-R
N_ENREGISTREMENTS = 3000     # ~5 h d'audio. Plus = meilleur mais plus long
DUREE_MAX_S = 12             # on ignore les enregistrements trop longs (mémoire GPU)

if OPTION == "A":
    MODELE_DEPART = "Professor/mms-300m-fongbe"
    TAUX_APPRENTISSAGE, EPOCHS = 3e-5, 5
else:
    MODELE_DEPART = "facebook/wav2vec2-xls-r-300m"
    TAUX_APPRENTISSAGE, EPOCHS = 3e-4, 12

DOSSIER_SORTIE = f"{BASE}/modeles/oreille-fon-kora-{OPTION}"
print(f"Option {OPTION} : on part de {MODELE_DEPART} · {EPOCHS} epochs · lr {TAUX_APPRENTISSAGE}")

## 2. Récupérer les enregistrements J'aime Ma Langue

Même méthode que le notebook 3 : on lit la première archive en streaming et on s'arrête quand on a assez de fichiers (environ 600 Mo à télécharger).

📚 *J'aime Ma Langue — JML-fon-speech-v1, CC-BY-SA-4.0 (MTDI, ASIN, IIDiA). À citer.*

In [ ]:
import requests, tarfile, pandas as pd
from huggingface_hub import hf_hub_download

DEPOT = "JaimeMaLangue/JML-fon-speech-v1"
DOSSIER_AUDIO = "/content/jml_train" if IN_COLAB else "./jml_train"
os.makedirs(DOSSIER_AUDIO, exist_ok=True)

transcriptions = pd.read_csv(hf_hub_download(DEPOT, "transcribed.csv", repo_type="dataset"))
voulus = set(transcriptions.loc[transcriptions["duration"] <= DUREE_MAX_S, "audio_path"])

recuperes = {}
url = f"https://huggingface.co/datasets/{DEPOT}/resolve/main/audio_transcribed_000.tar"
with requests.get(url, stream=True, timeout=60) as r:
    r.raise_for_status()
    r.raw.decode_content = True
    with tarfile.open(fileobj=r.raw, mode="r|") as archive:
        for membre in archive:
            if membre.isfile() and membre.name in voulus:
                chemin = os.path.join(DOSSIER_AUDIO, os.path.basename(membre.name))
                with open(chemin, "wb") as f:
                    f.write(archive.extractfile(membre).read())
                recuperes[membre.name] = chemin
                if len(recuperes) % 250 == 0:
                    print(f"  {len(recuperes)} fichiers", end="\r")
                if len(recuperes) >= N_ENREGISTREMENTS:
                    break

df = transcriptions[transcriptions["audio_path"].isin(recuperes)].copy()
df["fichier"] = df["audio_path"].map(recuperes)
print(f"\n✅ {len(df)} enregistrements · {df['duration'].sum()/3600:.1f} h · {df['speaker_id'].nunique()} voix")

## 3. Préparer le texte : l'alphabet du modèle

Un modèle vocal CTC écrit **lettre par lettre**. Il a donc un **alphabet** (son *vocabulaire*) : `a`, `ɖ`, `ɛ`, `é`, `|` pour l'espace, etc.

- **Option A** : on garde l'alphabet du modèle de départ. On écarte les phrases qui contiennent une lettre qu'il ne connaît pas (chiffres, ponctuation rare…).
- **Option B** : on **construit l'alphabet** à partir des textes de J'aime Ma Langue.

Les tons peuvent s'écrire de deux façons en Unicode : une lettre déjà accentuée (`é`, forme NFC) ou une lettre plus un accent séparé (`e` + `´`, forme NFD). On choisit automatiquement la forme qui correspond à l'alphabet.

In [ ]:
import re, json, unicodedata
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

def nettoyer(texte, forme="NFC"):
    t = unicodedata.normalize(forme, str(texte)).lower()
    t = re.sub(r"[\"“”«»,.;:!?()\[\]…\-–_/]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

if OPTION == "A":
    processor = Wav2Vec2Processor.from_pretrained(MODELE_DEPART)
    alphabet = set(processor.tokenizer.get_vocab())
    def couverture(forme):
        textes = df["transcription"].map(lambda t: nettoyer(t, forme).replace(" ", "|"))
        return textes.map(lambda t: all(c in alphabet for c in t)).mean()
    FORME = max(["NFC", "NFD"], key=couverture)
    print(f"Forme Unicode retenue : {FORME} ({couverture(FORME):.0%} des phrases compatibles)")
else:
    FORME = "NFC"
    lettres = sorted(set("".join(df["transcription"].map(nettoyer))) - {" "})
    vocab = {c: i for i, c in enumerate(lettres)}
    vocab["|"] = len(vocab); vocab["[UNK]"] = len(vocab); vocab["[PAD]"] = len(vocab)
    os.makedirs(DOSSIER_SORTIE, exist_ok=True)
    with open(f"{DOSSIER_SORTIE}/vocab.json", "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False)
    tok = Wav2Vec2CTCTokenizer(f"{DOSSIER_SORTIE}/vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")
    fe = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)
    processor = Wav2Vec2Processor(feature_extractor=fe, tokenizer=tok)
    alphabet = set(vocab)
    print(f"Alphabet de {len(vocab)} symboles :", "".join(lettres))

df["texte"] = df["transcription"].map(lambda t: nettoyer(t, FORME))
df = df[df["texte"].map(lambda t: len(t) > 0 and all(c in alphabet for c in t.replace(" ", "|")))]
df = df[~df["texte"].str.contains(r"\d")]
print(f"{len(df)} phrases utilisables")

## 4. Découpage honnête : par voix

Comme pour les messages (notebook 1), on évite la triche involontaire : les **voix** du test ne sont **jamais entendues** pendant l'entraînement. Le score mesure donc si le modèle comprend **une nouvelle personne**, ce qui est le vrai usage.

In [ ]:
import numpy as np
voix = df["speaker_id"].unique()
rng = np.random.default_rng(0)
rng.shuffle(voix)
voix_test, cumul = set(), 0
for v in voix:
    if cumul >= 0.1 * len(df):
        break
    voix_test.add(v); cumul += (df["speaker_id"] == v).sum()
test = df[df["speaker_id"].isin(voix_test)].reset_index(drop=True)
train = df[~df["speaker_id"].isin(voix_test)].reset_index(drop=True)
test = test.sample(min(len(test), 250), random_state=0).reset_index(drop=True)   # éval rapide
print(f"train {len(train)} ({train['speaker_id'].nunique()} voix) · test {len(test)} ({len(voix_test)} voix jamais entendues)")

## 5. L'entraînement

**Ce qui se passe, en une image :** le modèle écoute un enregistrement, écrit ce qu'il croit entendre, on compare au texte des linguistes, et il corrige ses paramètres. La méthode **CTC** lui permet d'apprendre sans qu'on lui dise à quel instant précis chaque lettre est prononcée.

On mesure le **WER** (taux d'erreur sur les mots) toutes les 400 étapes, et on garde la meilleure version.

In [ ]:
import torch, jiwer, soundfile as sf
from dataclasses import dataclass
from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer

class Voix(torch.utils.data.Dataset):
    def __init__(self, table):
        self.t = table
    def __len__(self):
        return len(self.t)
    def __getitem__(self, i):
        son, sr = sf.read(self.t["fichier"][i], dtype="float32")
        if son.ndim > 1:
            son = son.mean(axis=1)
        return {"son": son, "texte": self.t["texte"][i]}

@dataclass
class Assembleur:
    processor: Wav2Vec2Processor
    def __call__(self, lot):
        entree = self.processor.feature_extractor([x["son"] for x in lot], sampling_rate=16000,
                                                  padding=True, return_tensors="pt", return_attention_mask=True)
        etiq = self.processor.tokenizer([x["texte"] for x in lot], padding=True, return_tensors="pt")
        labels = etiq["input_ids"].masked_fill(etiq["attention_mask"].ne(1), -100)
        return {"input_values": entree["input_values"], "attention_mask": entree["attention_mask"], "labels": labels}

if OPTION == "A":
    modele = Wav2Vec2ForCTC.from_pretrained(MODELE_DEPART, layerdrop=0.0, mask_time_prob=0.05)
else:
    modele = Wav2Vec2ForCTC.from_pretrained(
        MODELE_DEPART, attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0, layerdrop=0.05,
        mask_time_prob=0.05, ctc_loss_reduction="mean",
        pad_token_id=processor.tokenizer.pad_token_id, vocab_size=len(processor.tokenizer))
modele.freeze_feature_encoder()   # la toute première couche (qui "entend" le signal brut) ne bouge pas

def vers_ids(logits, labels):
    # on garde seulement la lettre la plus probable à chaque instant (économise beaucoup de mémoire)
    return logits.argmax(dim=-1)

def mesurer(pred):
    ids = pred.predictions
    ids[ids == -100] = processor.tokenizer.pad_token_id
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    predit = processor.batch_decode(ids)
    attendu = processor.batch_decode(pred.label_ids, group_tokens=False)
    return {"wer": jiwer.wer(attendu, predit), "cer": jiwer.cer(attendu, predit)}

args = TrainingArguments(
    output_dir="/content/oreille_entrainement" if IN_COLAB else "./oreille_entrainement",
    per_device_train_batch_size=8, gradient_accumulation_steps=2, per_device_eval_batch_size=8,
    learning_rate=TAUX_APPRENTISSAGE, warmup_steps=300, num_train_epochs=EPOCHS,
    fp16=True, gradient_checkpointing=True,
    eval_strategy="steps", eval_steps=400, save_strategy="steps", save_steps=400, save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    logging_steps=50, report_to="none", remove_unused_columns=False, dataloader_num_workers=2,
)
trainer = Trainer(model=modele, args=args, train_dataset=Voix(train), eval_dataset=Voix(test),
                  data_collator=Assembleur(processor), compute_metrics=mesurer,
                  preprocess_logits_for_metrics=vers_ids)

avant = trainer.evaluate() if OPTION == "A" else None
if avant:
    print(f"WER AVANT adaptation (voix jamais entendues) : {avant['eval_wer']:.1%}")
trainer.train()
apres = trainer.evaluate()
print(f"WER APRÈS entraînement (voix jamais entendues) : {apres['eval_wer']:.1%}")

## 6. Sauvegarder et comparer

In [ ]:
trainer.save_model(DOSSIER_SORTIE)
processor.save_pretrained(DOSSIER_SORTIE)
resultat = {"option": OPTION, "modele_depart": MODELE_DEPART, "heures_audio": round(train["duration"].sum() / 3600, 2),
            "voix_train": int(train["speaker_id"].nunique()), "voix_test": len(voix_test),
            "wer_avant": avant["eval_wer"] if avant else None, "wer_apres": apres["eval_wer"], "cer_apres": apres["eval_cer"]}
with open(f"{BASE}/resultats/oreille_fon_entrainement_{OPTION}.json", "w") as f:
    json.dump(resultat, f, indent=2)
print("💾", DOSSIER_SORTIE)
print(json.dumps(resultat, indent=2))

In [ ]:
# Quelques exemples sur des voix jamais entendues
from transformers import pipeline
oreille = pipeline("automatic-speech-recognition", model=DOSSIER_SORTIE, device=0)
for i in range(5):
    son, _ = sf.read(test["fichier"][i], dtype="float32")
    print("attendu :", test["texte"][i])
    print("entendu :", oreille({"raw": son, "sampling_rate": 16000})["text"], "\n")

## ✅ Récapitulatif

- `modeles/oreille-fon-kora-A/` ou `-B/` : **votre** oreille fon entraînée. Le notebook 4 l'utilise automatiquement si elle existe.
- `resultats/oreille_fon_entrainement_A.json` : le WER avant et après, sur des **voix jamais entendues**.

### 🎤 Au jury
> **Option A** : « Nous avons adapté le modèle fon à la voix au téléphone grâce à J'aime Ma Langue : le taux d'erreur passe de X % à Y % sur des voix que le modèle n'avait jamais entendues. »
>
> **Option B** : « Nous avons entraîné notre propre reconnaissance vocale fon, sous licence commerciale, à partir de Z heures de J'aime Ma Langue. C'est la base de notre version commerciale. »